Sistema de razonamiento difuso - riesgo de llegar tarde a clase

Sistema hecho con SKFuzzy, metodo Mandami. Calcula el riesgo de llegar tarde segun el trafico que hay y el tiempo que falta para que empiece la clase. Tiene 9 reglas.

In [ ]:
!pip install scikit-fuzzy ipywidgets -q
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, clear_output

Variables linguisticas:

- Trafico [0,10]: bajo, medio, alto
- Tiempo restante en minutos [0,60]: poco, medio, mucho
- Riesgo de llegar tarde [0,10]: bajo, medio, alto

In [ ]:
trafico = ctrl.Antecedent(np.arange(0, 11, 1), 'trafico')
tiempoRestante = ctrl.Antecedent(np.arange(0, 61, 1), 'tiempoRestante')
riesgo = ctrl.Consequent(np.arange(0, 11, 1), 'riesgo')

trafico['bajo'] = fuzz.trapmf(trafico.universe, [0, 0, 2, 4])
trafico['medio'] = fuzz.trimf(trafico.universe, [3, 5, 7])
trafico['alto'] = fuzz.trapmf(trafico.universe, [6, 8, 10, 10])

tiempoRestante['poco'] = fuzz.trapmf(tiempoRestante.universe, [0, 0, 10, 20])
tiempoRestante['medio'] = fuzz.trimf(tiempoRestante.universe, [15, 30, 45])
tiempoRestante['mucho'] = fuzz.trapmf(tiempoRestante.universe, [40, 50, 60, 60])

riesgo['bajo'] = fuzz.trimf(riesgo.universe, [0, 0, 4])
riesgo['medio'] = fuzz.trimf(riesgo.universe, [3, 5, 7])
riesgo['alto'] = fuzz.trimf(riesgo.universe, [6, 10, 10])

Reglas (una por cada combinacion de trafico y tiempo restante, 3x3 = 9 reglas)

In [ ]:
reglas = [
  ctrl.Rule(trafico['bajo'] & tiempoRestante['mucho'], riesgo['bajo']),
  ctrl.Rule(trafico['bajo'] & tiempoRestante['medio'], riesgo['bajo']),
  ctrl.Rule(trafico['bajo'] & tiempoRestante['poco'], riesgo['medio']),
  ctrl.Rule(trafico['medio'] & tiempoRestante['mucho'], riesgo['bajo']),
  ctrl.Rule(trafico['medio'] & tiempoRestante['medio'], riesgo['medio']),
  ctrl.Rule(trafico['medio'] & tiempoRestante['poco'], riesgo['alto']),
  ctrl.Rule(trafico['alto'] & tiempoRestante['mucho'], riesgo['medio']),
  ctrl.Rule(trafico['alto'] & tiempoRestante['medio'], riesgo['alto']),
  ctrl.Rule(trafico['alto'] & tiempoRestante['poco'], riesgo['alto']),
]

sistemaControl = ctrl.ControlSystem(reglas)
print("Reglas cargadas:", len(reglas))

Interfaz grafica

Se mueven las barras deslizantes y se le da click a Calcular.

In [ ]:
sliderTrafico = widgets.FloatSlider(value=5, min=0, max=10, step=0.5, description='Trafico:')
sliderTiempo = widgets.FloatSlider(value=30, min=0, max=60, step=1, description='Min restantes:')

botonCalcular = widgets.Button(description='Calcular riesgo', button_style='success')
salida = widgets.Output()

def alCalcular(b):
  with salida:
    clear_output()
    simulador = ctrl.ControlSystemSimulation(sistemaControl)
    simulador.input['trafico'] = sliderTrafico.value
    simulador.input['tiempoRestante'] = sliderTiempo.value
    simulador.compute()

    resultado = simulador.output['riesgo']
    print(f"Trafico: {sliderTrafico.value}  Minutos restantes: {sliderTiempo.value}")
    print(f"Riesgo de llegar tarde: {resultado:.2f} de 10")

botonCalcular.on_click(alCalcular)

display(sliderTrafico, sliderTiempo, botonCalcular, salida)